# 01 — Quaternion Intuition

Quaternions are one of those mathematical objects that look intimidating at first but become intuitive once you connect them to geometry.

In this notebook we build that intuition step by step.

---

## What is a quaternion?

A quaternion is a four-dimensional number of the form:

$$q = w + xi + yj + zk$$

where $w, x, y, z \in \mathbb{R}$ and the imaginary units satisfy:

$$i^2 = j^2 = k^2 = ijk = -1$$

---

## Why quaternions for quantum computing?

The single-qubit state space is the **Bloch sphere** — a 2-sphere in 3D space.  Quaternions are the natural language for rotations on that sphere via **SU(2)**, and they avoid the *gimbal lock* problems of Euler angles.

`rqm-core` uses quaternions as the internal representation for all single-qubit operations.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from helpers.notebook_style import setup_notebook
setup_notebook()
print("Setup complete.")


## The unit quaternion as a rotation

A **unit quaternion** ($|q| = 1$) encodes a rotation in 3D:

$$q = \cos\!\left(\frac{\theta}{2}\right) + \sin\!\left(\frac{\theta}{2}\right)\,(n_x i + n_y j + n_z k)$$

where $\hat{n} = (n_x, n_y, n_z)$ is the rotation axis and $\theta$ is the rotation angle.

This is directly analogous to the SU(2) rotation matrix used to rotate qubits on the Bloch sphere.


In [ ]:
def unit_quaternion(theta, axis):
    """
    Build a unit quaternion for rotation by `theta` radians around `axis`.
    Returns (w, x, y, z).
    NOTE: In production code use rqm-core for this.
          This cell is illustrative only.
    """
    axis = np.array(axis, dtype=float)
    axis = axis / np.linalg.norm(axis)
    w = np.cos(theta / 2)
    xyz = np.sin(theta / 2) * axis
    return w, xyz[0], xyz[1], xyz[2]

# Example: 90-degree rotation around the Z axis
q = unit_quaternion(np.pi / 2, [0, 0, 1])
print(f"q = {q[0]:.4f} + {q[1]:.4f}i + {q[2]:.4f}j + {q[3]:.4f}k")
print(f"|q| = {np.sqrt(sum(v**2 for v in q)):.6f}  (should be 1.0)")


## Quaternion multiplication is rotation composition

If $q_1$ rotates by $\theta_1$ and $q_2$ rotates by $\theta_2$, then $q_2 q_1$ applies $q_1$ first, then $q_2$.

This is why quaternions are the natural tool for composing quantum gate sequences.


In [ ]:
def quat_mul(q1, q2):
    """Hamilton product of two quaternions (w, x, y, z)."""
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    return (
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2,
    )

# Two 90-degree Z rotations compose to a 180-degree Z rotation
q_90  = unit_quaternion(np.pi / 2, [0, 0, 1])
q_180 = quat_mul(q_90, q_90)
print("90° Z twice:")
print(f"  w={q_180[0]:.4f}, x={q_180[1]:.4f}, y={q_180[2]:.4f}, z={q_180[3]:.4f}")
print()
# Compare with a direct 180-degree rotation
q_direct_180 = unit_quaternion(np.pi, [0, 0, 1])
print("Direct 180° Z:")
print(f"  w={q_direct_180[0]:.4f}, x={q_direct_180[1]:.4f}, "
      f"y={q_direct_180[2]:.4f}, z={q_direct_180[3]:.4f}")


## Double cover: q and −q represent the same rotation

One subtlety of quaternions: both $q$ and $-q$ represent **the same physical rotation**.  
This is the famous **double cover** of SO(3) by SU(2), and it explains the global phase freedom in quantum mechanics.


In [ ]:
# Visualise a parameterised family of quaternions (rotation path)
from helpers.plotting import draw_bloch_sphere, plot_rotation_path

# Rotate the |0> state (north pole) around the Y axis
def bloch_from_z_rotation_y(theta):
    """Rotate (0,0,1) around Y axis by theta. Returns Bloch vector."""
    return np.array([np.sin(theta), 0, np.cos(theta)])

thetas = np.linspace(0, 2 * np.pi, 80)
path = np.array([bloch_from_z_rotation_y(t) for t in thetas])

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection="3d")
plot_rotation_path(path, ax=ax, title="Y-rotation of |0⟩ (full 360°)")
plt.tight_layout()
plt.show()


## Summary

- A quaternion $q = w + xi + yj + zk$ encodes a 3D rotation when $|q| = 1$.
- The rotation angle is $\theta = 2\arccos(w)$.
- Quaternion multiplication = rotation composition.
- $q$ and $-q$ give the same rotation (double cover / global phase).

**Next:** [02_spinor_to_bloch.ipynb](02_spinor_to_bloch.ipynb) — connecting quantum state vectors to the Bloch sphere.
